In [10]:
from ssb_experimental import *
from utils.universal_data_loader import *

# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m", "max_dpd_12m"],            
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m", "payment_ratio_avg_6m", "payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m", "utilization_avg_6m", "utilization_avg_12m", "utilization_max_12m"],
# }

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000, 1000, 500],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="Default_Status",
    n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.5,
    min_events = 100,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=False,
    max_feature_reuse=2,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None, #business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/Strategic_Segment_Builder/Datasets/credit_card_default_2.5M.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-09 14:54:38,560 | INFO     | [universal_data_loader.py:74] | Detecting local file format handler for extension: '.parquet'...
2026-07-09 14:54:38,601 | INFO     | [ssb_experimental.py:328] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-09 14:54:38,648 | INFO     | [ssb_experimental.py:343] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-09 14:54:38,650 | INFO     | [ssb_experimental.py:364] | Iteration 1 | Remaining Volume: 250,000 | Base Rate: 26.02%
2026-07-09 14:54:44,162 | INFO     | [ssb_experimental.py:500] | Feature Usage Tracker Update -> 'Housing_Status' used count = 1
2026-07-09 14:54:44,163 | INFO     | [ssb_experimental.py:500] | Feature Usage Tracker Update -> 'Employment_Type' used count = 1
2026-07-09 14:54:44,163 | INFO     | [ssb_experimental.py:500] | Feature Usage Tracker Update -> 'Missed_Payment_Pattern' used count = 1
2026-07-09 14:54:44,164 | INFO     | [ssb_experimental.py:515] | Segment 1 Captured (Size Floor: 1000 | Lift Floor

In [11]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |     capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
|   0.0   |   211725.0  |    45903.0    |  21.6804817569961  | 26.024399999999996 |        84.69        | 0.8330828667326088 |
|   1.0   |    1303.0   |     1295.0    | 99.38603223330774  | 26.024399999999996 |        0.5212       | 3.818955758184925  |
|   2.0   |   36229.0   |    17142.0    | 47.315686328631756 | 26.024399999999996 |  14.491599999999998 | 1.8181278465068076 |
|   3.0   |    743.0    |     721.0     | 97.03903095558546  | 26.024399999999996 | 0.29719999999999996 | 3.7287711130933077 |
+---------+-------------+---------------+--------------------+--------------------+---------------------+------

In [12]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Housing_Status=<ArrowStringArray>
['Other', 'Rent']
Length: 2, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str & Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str
SQL Filter: Housing_Status IN ('Other', 'Rent') AND Employment_Type IN ('Unemployed') AND Missed_Payment_Pattern IN ('Frequent')
--------------------------------------------------
Segment ID: 2
Raw Rule:   Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str
SQL Filter: Missed_Payment_Pattern IN ('Frequent')
--------------------------------------------------
Segment ID: 3
Raw Rule:   Housing_Status=<ArrowStringArray>
['Rent']
Length: 1, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str & Prior_Bankruptcy=<ArrowStringArray>
['Yes']
Length: 1, dtype: str
SQL Filter: Housing_Status IN ('Rent') AND Employment_Type IN ('Unemployed') AND Prior_Bankruptcy IN 

In [13]:
pd.DataFrame(final_eval)

,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,211725,45903.0,21.680482,26.0244,84.6900,0.833083
1,1,1303,1295.0,99.386032,26.0244,0.5212,3.818956
2,2,36229,17142.0,47.315686,26.0244,14.4916,1.818128
3,3,743,721.0,97.039031,26.0244,0.2972,3.728771


In [33]:
builder.explain_feature_journey("Prior_Bankruptcy")

AUDIT TRAIL FOR FEATURE: 'Prior_Bankruptcy'

[Iteration 1]
  • Current dynamic IV   : 5.2008
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : Housing_Status=<ArrowStringArray>
['Other', 'Rent']
Length: 2, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str & Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str (Variables: ['Housing_Status', 'Employment_Type', 'Missed_Payment_Pattern'])

[Iteration 2]
  • Current dynamic IV   : 5.3074
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str (Variables: ['Missed_Payment_Pattern'])

[Iteration 3]
  • Current dynamic IV   : 5.9548
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  🎉 SELECTED as part of winning rule!
     Rule: Housing_Status=<Arrow

In [15]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN Housing_Status IN ('Other', 'Rent') AND Employment_Type IN ('Unemployed') AND Missed_Payment_Pattern IN ('Frequent')
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN Missed_Payment_Pattern IN ('Frequent')
            THEN 1 ELSE 0 END AS seg_2,
            CASE WHEN Housing_Status IN ('Rent') AND Employment_Type IN ('Unemployed') AND Prior_Bankruptcy IN ('Yes')
            THEN 1 ELSE 0 END AS seg_3,
            FROM predicted
""").df()
conn.close()

In [ ]:
scorer = StrategicSegmentScore(
    target_col="Default_Status",
    primary_key="Customer_ID",
    segment_cols=["seg_1", "seg_2", "seg_3"],
)

In [19]:
scorer.calculate_and_export_weights(predicted, export_path="scorecard_model_test1.json")

2026-07-09 14:57:02,924 | INFO     | [ssb_experimental.py:626] | Initializing DuckDB scorecard engine...
2026-07-09 14:57:03,081 | INFO     | [ssb_experimental.py:661] | Computing harmonic scorecard weights...
2026-07-09 14:57:03,081 | INFO     | [ssb_experimental.py:698] | Scoring training dataset via NumPy Linear Algebra engine...
2026-07-09 14:57:03,093 | INFO     | [ssb_experimental.py:715] | Dataset Zero-Inflation Rate: 73.98%
2026-07-09 14:57:03,097 | INFO     | [ssb_experimental.py:721] | Normal Distribution (<80%). Deciling across full dataset...
2026-07-09 14:57:03,097 | INFO     | [ssb_experimental.py:730] | Calibrating deciles across 250,000 target customers...


{'model_metadata': {'total_training_population': 250000,
  'active_scored_population': 250000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2602},
 'segment_weights': {'seg_1': {'weight': 15,
   'lift': 3.819,
   'response_rate': 0.9939,
   'capture_rate': 0.0199},
  'seg_2': {'weight': 68,
   'lift': 1.8876,
   'response_rate': 0.4912,
   'capture_rate': 0.2834},
  'seg_3': {'weight': 10,
   'lift': 3.748,
   'response_rate': 0.9754,
   'capture_rate': 0.0134}},
 'decile_min_thresholds': {'1': 68,
  '2': 0,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [20]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/scorecard_model_test1.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-09 14:57:20,167 | INFO     | [2422015793.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/scorecard_model_test1.json...


In [21]:
model_artifact.get("decile_min_thresholds")

{'1': 68,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [22]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 15
Segment: seg_2 | Weight: 68
Segment: seg_3 | Weight: 10


In [23]:
model_artifact.get("decile_min_thresholds")

{'1': 68,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [25]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 15 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 68 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 10 ELSE 0 END AS seg_3_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=68 THEN 1
                    WHEN total_weight >= 0 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [26]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Default_Status) AS events, 
                    (SUM(Default_Status)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()

In [27]:
scored

,decile_band,count,events,response_rate
0,1,37532,18437.0,49.123415
1,2,212468,46624.0,21.944010
